# Testing Bio Portal's RESTFUL API

https://data.bioontology.org/documentation#nav_resource_endpoints


"""
What is a .owl file?
An OWL file (Web Ontology Language) is an XML-based file format used to represent ontologies - formal descriptions of knowledge domains. 
"""

In [ ]:
"""
classes_search_terms.txt

heart
lung
experiment
human
brain
melanoma

"""



In [ ]:

# CLASS SEARCH
# https://github.com/ncbo/ncbo_rest_sample_code/blob/master/python/python3/classes_search.py

import urllib.request, urllib.error, urllib.parse
import json
import os
from pprint import pprint
from dotenv import load_dotenv

REST_URL = "http://data.bioontology.org"

# Create a .env file with your API key
# Format: BIO_PORTAL_API_KEY=your_api_key_here
load_dotenv()
API_KEY = os.environ.get("BIO_PORTAL_API_KEY", "")


def get_json(url):
    opener = urllib.request.build_opener()
    opener.addheaders = [('Authorization', 'apikey token=' + API_KEY)]
    return json.loads(opener.open(url).read())

# Get list of search terms
# path = os.path.join(os.path.dirname(__file__), 'classes_search_terms.txt')
# terms_file = open(path, "r")
# terms = []
# for line in terms_file:
#     terms.append(line)
terms = ['glaucoma']

# Do a search for every term
search_results = []
for term in terms:
    search_results.append(get_json(REST_URL + "/search?q=" + term)["collection"])

# Print the results
for result in search_results:
    pprint(result)


In [ ]:
# Explore the DOID (Human Disease Ontology) hierarchy

import urllib.request, urllib.error, urllib.parse
import json
from pprint import pprint
from dotenv import load_dotenv
import os

REST_URL = "http://data.bioontology.org"

# Load API key from .env file
load_dotenv()
API_KEY = os.environ.get("BIO_PORTAL_API_KEY", "")

def get_json(url):
    opener = urllib.request.build_opener()
    opener.addheaders = [('Authorization', 'apikey token=' + API_KEY)]
    return json.loads(opener.open(url).read())

# Access the DOID ontology specifically
doid_acronym = "DOID"  # Human Disease Ontology
doid_url = f"{REST_URL}/ontologies/{doid_acronym}"
doid_ontology = get_json(doid_url)
print(f"Accessing DOID: {doid_ontology['name']}")


In [ ]:

# Get the root classes of the DOID ontology
roots_url = doid_ontology['links']['roots']
roots = get_json(roots_url)
print("\nRoot classes in DOID:")
if len(roots) > 1:
    for root in roots:
        print(f"- {root['prefLabel']}")
        if root['prefLabel'] == 'disease':
            disease_root = root
            
        
elif len(roots) == 0: 
    print("No roots in DOID")
else: 
    print("\tSINGLE ROOT: ", roots['prefLabel']) # type(root) == dict
    print()


In [ ]:
disease_root

In [ ]:
# CORRECTED: Get ALL children using the proper nextPage pattern from BioPortal sample code
# Based on: https://github.com/ncbo/ncbo_rest_sample_code
# Pattern matches the official sample: check nextPage link to determine if there are more pages

def get_all_children_proper(children_url):
    """
    Get ALL children using the proper BioPortal API pagination pattern.
    Uses the 'nextPage' link from the response, exactly like the official sample code.
    
    The key insight: The API response includes a 'links.nextPage' field that tells you
    if there's another page. When you hit the last page, nextPage will be None/empty.
    
    Args:
        children_url: The URL to the children endpoint
    
    Returns:
        List of all child objects
    """
    all_children = []
    
    # Get the first page
    page = get_json(children_url)
    
    # print(f"Fetching children from: {children_url}")
    page_num = 1
    
    # Iterate over the available pages adding children from all pages
    # When we hit the last page, nextPage will be None/empty and the loop will exit
    # This matches the pattern from the official sample code
    next_page = page.get("links", {}).get("nextPage")
    
    # Process first page
    if 'collection' in page:
        all_children.extend(page['collection'])
        # print(f"Page {page_num}: Found {len(page['collection'])} items (Total: {len(all_children)})")
    
    # Continue while there's a next page
    # IMPORTANT: Check next_page BEFORE fetching, just like the sample code
    while next_page:
        page_num += 1
        page = get_json(next_page)
        
        if 'collection' in page:
            all_children.extend(page['collection'])
            # print(f"Page {page_num}: Found {len(page['collection'])} items (Total: {len(all_children)})")
        
        # Check for next page - this is the KEY part from the sample code
        # When we hit the last page, nextPage will be None/empty
        next_page = page.get("links", {}).get("nextPage")
    
    # print(f"\n✅ Reached last page. Total children: {len(all_children)}")
    return all_children

# Now use this function to get ALL children of disease
if disease_root is None:
    print("Disease root not found. Run Cell 4 first.")
else:
    # Get the children URL
    disease_root_info = get_json(disease_root['links']['self'])
    children_url = disease_root_info['links']['children']
    
    print("=" * 60)
    print("GETTING ALL CHILDREN (using nextPage pattern)")
    print("=" * 60)
    
    # Get ALL children using proper pagination
    all_disease_children = get_all_children_proper(children_url)
    
    print("\n" + "=" * 60)
    print(f"✅ TOTAL: {len(all_disease_children)} direct children of 'disease'")
    print("=" * 60)
    
    # Store all children
    l1_disease_children = []
    for child in all_disease_children:
        child_info = {
            'prefLabel': child['prefLabel'],
            'synonym': child.get('synonym', []),
            'definition': child.get('definition', []),
            'links': child.get('links', {}),
            '@id': child.get('@id', '')
        }
        l1_disease_children.append(child_info)
    
    # Display first 10 as sample
    print("\nFirst 10 children:")
    for i, child in enumerate(l1_disease_children[:10], 1):
        print(f"{i}. {child['prefLabel']} ({child['@id']})")


In [ ]:
term = "Nelson syndrome"
search_results = []
search_results.append(get_json(REST_URL + "/search?q=" + term)["collection"])

In [ ]:
# TRAVERSING THE TREE

"""
Algorithm Steps:

Initialize:
Create a list database = [] to store your rows.
Create a set visited_ids = set() to avoid processing the same disease twice (ontologies can have multiple links to the same node).
Create a queue and add the Root Node ("disease") to it.
Queue item format: (node_data, current_path_string, current_level_int)
"""

In [ ]:
from collections import deque
import time
import csv
import pandas as pd
import sys

# CONFIGURATION
OUTPUT_FILE = "disease_ontology_flattened.csv"
BATCH_SIZE = 100  # Save every N items
MAX_ITEMS = None  # Set to integer (e.g., 100) for testing, None for full run
RATE_LIMIT_SLEEP = 0.1  # Sleep between requests to respect API limits

def traverse_ontology_bfs(root_node):
    """
    Breadth-First Search traversal of the Disease Ontology.
    Captures: id, level, path, prefLabel for each node.
    """
    
    # 1. Initialize
    database = []
    visited_ids = set()
    queue = deque()
    
    # Start with the root
    # If root_node is not fully loaded (only has links), fetch it first
    if 'links' in root_node and 'self' in root_node['links']:
        print("Fetching full root info...")
        root_full = get_json(root_node['links']['self'])
    else:
        root_full = root_node
        
    # Add root to queue: (node_data, path, level)
    # Root path is just its label
    root_label = root_full.get('prefLabel', 'disease')
    root_path = root_label
    
    queue.append((root_full, root_path, 0))
    visited_ids.add(root_full['@id'])
    
    print(f"Starting BFS traversal from root: {root_label}")
    print(f"Output will be saved to: {OUTPUT_FILE}")
    
    processed_count = 0
    
    try:
        # 2. Loop while queue is not empty
        while queue:
            if MAX_ITEMS and processed_count >= MAX_ITEMS:
                print(f"\n🛑 Reached MAX_ITEMS limit ({MAX_ITEMS}). Stopping.")
                break
                
            # Pop from the left (FIFO for BFS)
            current_node, current_path, current_level = queue.popleft()
            
            # Record data
            node_id = current_node.get('@id', '')
            node_label = current_node.get('prefLabel', '')
            
            # Create record
            record = {
                'id': node_id,
                'level': current_level,
                'path': current_path,
                'prefLabel': node_label
            }
            database.append(record)
            processed_count += 1
            
            # Progress update
            if processed_count % 10 == 0:
                sys.stdout.write(f"\rProcessing item {processed_count} | Queue size: {len(queue)} | Level: {current_level} | Current: {node_label[:30]}...")
                sys.stdout.flush()
            
            # Save batch periodically
            if processed_count % BATCH_SIZE == 0:
                save_to_csv(database, OUTPUT_FILE)
                
            # Fetch Children
            # Check if node has children link
            if 'links' in current_node and 'children' in current_node['links']:
                children_url = current_node['links']['children']
                
                # Respect rate limits
                time.sleep(RATE_LIMIT_SLEEP)
                
                try:
                    # Use our robust pagination function
                    children = get_all_children_proper(children_url)
                    
                    # Process each child
                    for child in children:
                        child_id = child.get('@id')
                        
                        # Check if already visited
                        if child_id and child_id not in visited_ids:
                            visited_ids.add(child_id)
                            
                            # Create new path: parent/child
                            child_label = child.get('prefLabel', 'Unknown')
                            new_path = f"{current_path}/{child_label}"
                            
                            # Add to queue
                            queue.append((child, new_path, current_level + 1))
                            
                except Exception as e:
                    print(f"\n❌ Error fetching children for {node_label}: {str(e)}")
                    continue
    
    except KeyboardInterrupt:
        print("\n\n🛑 Traversal interrupted by user.")
        
    # Final save
    print(f"\n\n✅ Traversal complete. Processed {len(database)} nodes.")
    save_to_csv(database, OUTPUT_FILE)
    return database

def save_to_csv(data, filename):
    """Helper to save list of dicts to CSV"""
    if not data:
        return
        
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)
    # print(f"Saved {len(data)} records to {filename}")

# Start the traversal
# Only run if we have the disease root
if disease_root:
    print("Starting traversal... This may take a while.")
    # Set MAX_ITEMS=100 for testing, set to None for full run
    # I'll set it to 200 first to verify it works correctly across multiple levels
    global MAX_ITEMS
    MAX_ITEMS = 200 
    
    ontology_data = traverse_ontology_bfs(disease_root)
    
    # Show sample output
    print("\nSample Output:")
    print(pd.DataFrame(ontology_data).head(10))
else:
    print("Disease root not found. Please run previous cells.")